# BLACKBOX · MCP-first WebGPU smoke on Colab

This notebook adapts the supplied headless-Chrome/WebGPU setup to **BLACKBOX**. It keeps **`chrome-devtools-mcp` as the browser operator**: the host preflight uses OS tools; page creation, navigation, JavaScript evaluation, console capture, and screenshot capture are MCP calls.

**First:** in Colab choose **Runtime → Change runtime type → T4 GPU → Save**, then connect. Availability depends on your Colab account/session. The notebook is designed to degrade honestly if no GPU is attached.

This is a **known-answer compute smoke test**, not an ML model benchmark: it dispatches `out[i] = in[i] * 2 + 1` for 1,024 floats, reads the output back, and checks every value. A WebGPU pass is not alone proof of physical T4 execution; compare host `nvidia-smi`/Vulkan evidence with browser adapter metadata in the report.

## Why this differs from running Puppeteer directly

BLACKBOX already has an audited MCP client, wire log, tool discovery/schema validation, and evidence/report pipeline. This notebook reuses that project instead of adding a second browser-control stack. The regular performance mission remains unchanged and GPU-disabled; the WebGPU lane is opt-in.


## 1 · Upload the BLACKBOX project ZIP

Upload the ZIP delivered with this project. The archive has a top-level `blackbox/` directory. Dependencies and Chrome binaries are excluded from the ZIP and installed in the next cells.


In [ ]:
from pathlib import Path
import zipfile

PROJECT = Path('/content/blackbox')
if not (PROJECT / 'package.json').exists():
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise RuntimeError('Please upload the BLACKBOX project .zip file.')
    archive = Path('/content') / zip_names[0]
    with zipfile.ZipFile(archive) as zf:
        # Reject path traversal and unexpectedly huge archives before extracting.
        root = Path('/content').resolve()
        members = zf.infolist()
        total_bytes = sum(info.file_size for info in members)
        if total_bytes > 250 * 1024 * 1024:
            raise RuntimeError(f'Project ZIP expands to {total_bytes:,} bytes; refusing extraction.')
        for info in members:
            target = (root / info.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe archive path: {info.filename}')
        # The supplied project archive contains blackbox/ at its root.
        zf.extractall(root)
    if not (PROJECT / 'package.json').exists():
        raise RuntimeError('ZIP extracted, but /content/blackbox/package.json was not found.')

print('Project:', PROJECT)
print('Notebook:', 'MCP-first WebGPU compute smoke')
print('Existing run artifacts:', len(list((PROJECT / 'runs').glob('*'))) if (PROJECT / 'runs').exists() else 0)


## 2 · Attest the Colab runtime

Install only the Vulkan **diagnostic CLI**. Colab supplies its own NVIDIA driver/runtime; do not replace it with a hard-coded driver package. `nvidia-smi` and `vulkaninfo` are separate host-side evidence, not browser results.


In [ ]:
import subprocess, shutil, textwrap

def run_cmd(args, check=False):
    print('$', ' '.join(args))
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[:12000] if p.stdout else '(no output)')
    if check and p.returncode:
        raise RuntimeError(f'Command exited {p.returncode}: {args[0]}')
    return p

run_cmd(['apt-get', 'update', '-qq'])
run_cmd(['apt-get', 'install', '-y', '-qq', 'vulkan-tools'], check=True)
if shutil.which('nvidia-smi'):
    gpu_probe = run_cmd(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'])
    if 't4' in gpu_probe.stdout.lower():
        print('T4 target: confirmed by nvidia-smi (still check browser adapter evidence).')
    else:
        print('T4 target: not confirmed; do not label this as a T4 run.')
else:
    print('nvidia-smi not found: this runtime has no NVIDIA CLI evidence; do not label this a T4 run.')
if shutil.which('vulkaninfo'):
    run_cmd(['vulkaninfo', '--summary'])
else:
    print('vulkaninfo not found.')


## 3 · Install the pinned browser/MCP toolchain

The shared setup pins **Node 22.20.0**, **Chrome for Testing 159.0.8129.0**, and `chrome-devtools-mcp@1.13.0`. It installs Vulkan tools only; it does not install an NVIDIA kernel driver. Chrome/MCP compatibility and the launch flags are captured in the artifacts.


In [ ]:
import os, subprocess

setup_env = os.environ.copy()
setup_env['BLACKBOX_GPU_TOOLS'] = '1'
subprocess.run(['bash', 'scripts/setup.sh'], cwd=PROJECT, env=setup_env, check=True)

NODE = Path.home() / '.cache/rt/node-v22.20.0-linux-x64/bin/node'
CHROME = Path.home() / '.cache/chrome-159/chrome-linux64/chrome'
if not NODE.exists() or not CHROME.exists():
    raise RuntimeError(f'Pinned runtime not found: Node={NODE.exists()} Chrome={CHROME.exists()}')
os.environ['CHROME_PATH'] = str(CHROME)
os.environ['PATH'] = str(NODE.parent) + os.pathsep + os.environ.get('PATH', '')
print('Node:', subprocess.check_output([str(NODE), '--version'], text=True).strip())
print('Chrome:', subprocess.check_output([str(CHROME), '--version'], text=True).strip())


## 4 · Start BLACKBOX's local control plane

The test page is served from `localhost`, a secure context for WebGPU. The headless browser is a separate Chrome launched by the MCP server; the notebook kernel only starts the app server and the runner.


In [ ]:
import os, subprocess, time, urllib.request

server_env = os.environ.copy()
server_env['CHROME_PATH'] = str(CHROME)
server_env['PATH'] = str(NODE.parent) + os.pathsep + server_env.get('PATH', '')
server_log = open('/content/blackbox-server.log', 'w')
server = subprocess.Popen([str(NODE), 'server.mjs'], cwd=PROJECT, env=server_env,
                          stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)

ready = False
for _ in range(60):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/api/health', timeout=2) as r:
            print('Control plane:', r.read().decode())
            ready = True
            break
    except Exception:
        time.sleep(1)
if not ready:
    server_log.flush()
    print(Path('/content/blackbox-server.log').read_text()[-8000:])
    raise RuntimeError('BLACKBOX server did not become ready on port 8080.')
print('GPU fixture:', 'http://127.0.0.1:8080/lab/webgpu/')


## 5 · Run the WebGPU smoke through MCP

All browser actions are made by the official MCP server. The runner records `wire.jsonl`, host preflight, a screenshot, and a self-contained HTML report. The single timing is submit-plus-readback wall time—**not** a kernel-only benchmark.


In [ ]:
import json, subprocess

run_env = os.environ.copy()
run_env['CHROME_PATH'] = str(CHROME)
run_env['PATH'] = str(NODE.parent) + os.pathsep + run_env.get('PATH', '')
cmd = [str(NODE), 'agent/webgpu-smoke.mjs', '--origin', 'http://127.0.0.1:8080']
completed = subprocess.run(cmd, cwd=PROJECT, env=run_env, text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode:
    print('Runner exit code:', completed.returncode, '(inspect the saved report; host/browser GPU support may be unavailable)')

try:
    cli_result = json.loads(completed.stdout) if completed.stdout.strip().startswith('{') else {}
except Exception:
    cli_result = {}
run_id = cli_result.get('runId')
if not run_id:
    candidates = sorted(PROJECT.glob('runs/gpu-*/summary.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise RuntimeError('No GPU smoke summary was written.')
    summary_path = candidates[0]
    run_id = summary_path.parent.name
else:
    summary_path = PROJECT / 'runs' / run_id / 'summary.json'
summary = json.loads(summary_path.read_text())
print('Run:', run_id)
print('Verdict:', json.dumps(summary.get('verdict', {}), indent=2))
print('Report:', summary_path.parent / 'report.html')


## 6 · Review browser and host evidence

The report intentionally keeps the two evidence planes separate. If the smoke passes but the host preflight is missing, record it as **WebGPU pass / hardware unattested**, not “T4 verified.”


In [ ]:
from IPython.display import display, FileLink, Image, Markdown

print('Host GPU:', summary.get('hostPreflight', {}).get('hostGpu', {}).get('status'))
print('Hardware assessment:', summary.get('hardwareAssessment', {}))
print('WebGPU result:', json.dumps(summary.get('browserResult', {}).get('webgpu', {}), indent=2))
print('MCP:', json.dumps(summary.get('mcp', {}), indent=2))
report_path = summary_path.parent / 'report.html'
wire_path = summary_path.parent / 'wire.jsonl'
host_path = summary_path.parent / 'host-preflight.json'
for p in [report_path, summary_path, host_path, wire_path]:
    if p.exists(): display(FileLink(str(p)))
screenshot = summary.get('screenshot')
if screenshot and (summary_path.parent / screenshot).exists():
    display(Image(filename=str(summary_path.parent / screenshot)))


## 7 · Download the run bundle

The bundle contains only this run's artifacts (not Chrome binaries or `node_modules`). Preserve it with the notebook output if you want reproducible evidence.


In [ ]:
import zipfile
from google.colab import files

bundle = Path('/content') / f'blackbox-{run_id}-webgpu-evidence.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for item in summary_path.parent.rglob('*'):
        if item.is_file():
            zf.write(item, item.relative_to(summary_path.parent))
print('Created:', bundle, 'bytes:', bundle.stat().st_size)
files.download(str(bundle))


## Interpretation and next step

A successful smoke demonstrates that the selected browser context executed the WGSL transform and returned numerically correct values. It does not measure TensorFlow.js performance, VRAM, sustained throughput, or model quality. To extend this into an AI benchmark, pin a model and input set, compare backend output within a tolerance, separate load/compile/first-inference/warm-inference, use repeated p50/p95 timings, and keep GPU memory telemetry distinct from JS heap snapshots.

If this runtime is CPU-only, the report is still useful: it exposes which layer failed (host driver/device, Vulkan, browser API, adapter/device, shader/compute, or MCP/toolchain), rather than fabricating a hardware result.
